# 🔹 Notebook 2 — Redis Vector Search

**Obiettivo**: Testare operazioni di embedding, upsert e ricerca semantica KNN su Redis for AI.

**Prerequisiti**: Stack Docker attivo, Ollama con `nomic-embed-text` disponibile.


## Setup: connessione Redis + Ollama embedding

In [ ]:
import os, json, numpy as np
import redis
import httpx

REDIS_URL = os.getenv('REDIS_URL', 'redis://localhost:6379')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
EMBEDDING_MODEL = os.getenv('OLLAMA_EMBEDDING_MODEL', 'nomic-embed-text')
VECTOR_DIM = 768
INDEX_NAME = 'idx:kg_chunks'

r = redis.from_url(REDIS_URL, decode_responses=True)
print('Redis PING:', r.ping())

def get_embedding(text: str) -> list[float]:
    """Genera embedding 768D via Ollama HTTP API."""
    resp = httpx.post(
        f'{OLLAMA_BASE_URL}/api/embed',
        json={'model': EMBEDDING_MODEL, 'input': text},
        timeout=60.0
    )
    resp.raise_for_status()
    return resp.json()['embeddings'][0]

# Test embedding
test_emb = get_embedding('Knowledge Graph test')
print(f'Embedding dim: {len(test_emb)}, primi 5 valori: {test_emb[:5]}')

## 2.1 — Crea indice vettoriale RedisSearch

In [ ]:
from redis.commands.search.field import VectorField, TextField, NumericField, TagField
from redis.commands.search.indexDefinition import IndexDefinition, IndexType

# Drop indice se esiste
try:
    r.ft(INDEX_NAME).dropindex(delete_documents=True)
    print('Indice precedente rimosso.')
except:
    pass

# Definisci schema
schema = (
    TextField('text'),
    TextField('doc_name'),
    TagField('intent'),
    NumericField('page'),
    VectorField('embedding',
        'HNSW', {
            'TYPE': 'FLOAT32',
            'DIM': VECTOR_DIM,
            'DISTANCE_METRIC': 'COSINE'
        }
    )
)

definition = IndexDefinition(prefix=['chunk:'], index_type=IndexType.HASH)
r.ft(INDEX_NAME).create_index(schema, definition=definition)
print(f'Indice {INDEX_NAME} creato con successo.')

## 2.2 — Inserisci chunk di esempio con embedding

In [ ]:
import struct

sample_chunks = [
    {'id': 'chunk:001', 'text': 'La garanzia del prodotto SlimWash Pro copre difetti di fabbricazione per 24 mesi dalla data di acquisto.',
     'doc_name': 'manuale_slimwash.pdf', 'intent': 'warranty', 'page': 12},
    {'id': 'chunk:002', 'text': 'Il Knowledge Graph permette di navigare relazioni tra entita come persone, aziende e tecnologie.',
     'doc_name': 'kg_intro.pdf', 'intent': 'documentation', 'page': 1},
    {'id': 'chunk:003', 'text': 'Neo4j utilizza il linguaggio Cypher per interrogare grafi. Le query MATCH-RETURN sono la forma base.',
     'doc_name': 'neo4j_guide.pdf', 'intent': 'documentation', 'page': 5},
    {'id': 'chunk:004', 'text': 'Redis for AI supporta ricerca vettoriale KNN con indici HNSW e FLAT per embedding fino a 32768 dimensioni.',
     'doc_name': 'redis_vector.pdf', 'intent': 'documentation', 'page': 3},
    {'id': 'chunk:005', 'text': 'Il GDPR Art. 30 richiede un registro dei trattamenti dati personali con finalita, base giuridica e tempi di conservazione.',
     'doc_name': 'gdpr_compliance.pdf', 'intent': 'compliance', 'page': 30},
    {'id': 'chunk:006', 'text': 'La pipeline RAG combina ricerca vettoriale semantica con traversal del grafo per fornire contesto accurato all LLM.',
     'doc_name': 'rag_architecture.pdf', 'intent': 'documentation', 'page': 8},
]

for chunk in sample_chunks:
    emb = get_embedding(chunk['text'])
    emb_bytes = struct.pack(f'{len(emb)}f', *emb)
    r.hset(chunk['id'], mapping={
        'text': chunk['text'],
        'doc_name': chunk['doc_name'],
        'intent': chunk['intent'],
        'page': chunk['page'],
        'embedding': emb_bytes
    })
    print(f"  Inserito: {chunk['id']} ({chunk['doc_name']})")

print(f'\nTotale chunk inseriti: {len(sample_chunks)}')

## 2.3 — Ricerca semantica KNN

In [ ]:
from redis.commands.search.query import Query

def semantic_search(query_text: str, top_k: int = 3):
    """Esegue ricerca semantica KNN su Redis."""
    query_emb = get_embedding(query_text)
    query_bytes = struct.pack(f'{len(query_emb)}f', *query_emb)
    
    q = (
        Query(f'*=>[KNN {top_k} @embedding $vec AS score]')
        .sort_by('score')
        .return_fields('text', 'doc_name', 'intent', 'page', 'score')
        .dialect(2)
    )
    results = r.ft(INDEX_NAME).search(q, query_params={'vec': query_bytes})
    return results.docs

# Test: cerchiamo informazioni sulla garanzia
print('Query: "Qual e la garanzia del mio prodotto?"\n')
for doc in semantic_search('Qual e la garanzia del mio prodotto?', top_k=3):
    print(f'  Score: {float(doc.score):.4f}')
    print(f'  Fonte: {doc.doc_name} (pag. {doc.page})')
    print(f'  Testo: {doc.text[:120]}...')
    print()

## 2.4 — Ricerca semantica + filtro per intent

In [ ]:
# Ricerca solo nei documenti di compliance
print('Query: "obblighi registro trattamenti" (filtro: compliance)\n')

query_emb = get_embedding('obblighi registro trattamenti dati personali')
query_bytes = struct.pack(f'{len(query_emb)}f', *query_emb)

q = (
    Query(f'@intent:{{compliance}}=>[KNN 3 @embedding $vec AS score]')
    .sort_by('score')
    .return_fields('text', 'doc_name', 'intent', 'score')
    .dialect(2)
)
results = r.ft(INDEX_NAME).search(q, query_params={'vec': query_bytes})

for doc in results.docs:
    print(f'  [{doc.intent}] Score: {float(doc.score):.4f}')
    print(f'  {doc.text[:150]}')
    print()

## Cleanup

In [ ]:
# r.ft(INDEX_NAME).dropindex(delete_documents=True)
print('Notebook completato. Indice mantenuto per notebook successivi.')